# sklearn Course 1

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler, PolynomialFeatures
from sklearn.linear_model    import LinearRegression, Ridge, LogisticRegression
from sklearn.pipeline        import Pipeline
from sklearn.metrics         import mean_squared_error, r2_score, accuracy_score

## 2. Load Data

In [ ]:
df = pd.read_csv('your_file.csv')  
df.head()

## 3. Plot Features vs Target


In [ ]:
feature_cols = ['col1', 'col2', 'col3']    # all features
target_col   = 'target'                   # Our required feature

for col in feature_cols:
    plt.scatter(df[col], df[target_col], alpha=0.5)
    plt.xlabel(col)
    plt.ylabel(target_col)
    plt.title(f'{col} vs {target_col}')
    plt.show()


If we plot features and target and get the data points
1. Data looks like straight line use linear regression
2. Data looks like a curve use polynomial regression

## 4. Prepare X and y — Split Data

In [ ]:
X = df[feature_cols].values
y = df[target_col].values

# we use the randome state = 42 so this means that train test split the data randomly so if we fix it will not be random everytime
# we have 100% data we split it into two parts 70% training and 30 % testing
# we spilt that 30% again into 50 % meaning 15 and 15 % into two types of test
# one is x val and y val they are used to test repeatdly to check for overfitting etc 
# the test is the final test at the end
# we do not test on the x test repeadtly beacuse the model can get overfitted to it thats y we split the test as well
X_train, X_tmp,  y_train, y_tmp  = train_test_split(X, y, test_size=0.30, random_state=42)
X_val,   X_test, y_val,   y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=42)

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

---
# LINEAR REGRESSION

## 5a. Pipeline 

In [ ]:
pipe_linear = Pipeline([
    ('scale', StandardScaler()), # to scale the data so the range of the data become normal
    ('model', LinearRegression()), # linear regression for straight line
])

pipe_linear.fit(X_train, y_train)  # training the data

y_pred_train = pipe_linear.predict(X_train) # checking the prediction for the values it has seen and trained on
y_pred_val   = pipe_linear.predict(X_val)  # checking for the new values it has not seen

print('=== Linear Regression ===')
print(f'Train RMSE : {np.sqrt(mean_squared_error(y_train, y_pred_train)):.4f}') # it tells us the mean squared error 
print(f'Val   RMSE : {np.sqrt(mean_squared_error(y_val,   y_pred_val)):.4f}')
print(f'Val   R²   : {r2_score(y_val, y_pred_val):.4f}') # it tell us how good the model is relative to the data

## 6a. Plot Predicted vs Actual

In [ ]:
plt.scatter(y_val, y_pred_val, alpha=0.5)
plt.plot([y_val.min(), y_val.max()],
         [y_val.min(), y_val.max()], 'r--')   # perfect prediction line
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Linear Regression — Actual vs Predicted')
plt.show()

# dots close to red line = good predictions
# scattered far away    = model is off

## 7a. Check Overfitting / Underfitting

In [ ]:
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
val_rmse   = np.sqrt(mean_squared_error(y_val,   y_pred_val))

print(f'Train RMSE : {train_rmse:.4f}')
print(f'Val   RMSE : {val_rmse:.4f}')
print()

if train_rmse > 0.3 and val_rmse > 0.3:     # this is the threshold for checking the fitting
    print('→ HIGH BIAS (underfitting) — try polynomial features')
elif val_rmse > train_rmse * 1.5:
    print('→ HIGH VARIANCE (overfitting) — try Ridge regularization')
else:
    print('→ Good fit')

## 8a. Regularization — Ridge (if overfitting)

In [ ]:
# alpha bigger → stronger regularization → weights shrink
# try: 0.01, 0.1, 1.0, 10, 100

pipe_ridge = Pipeline([
    ('scale', StandardScaler()),
    ('model', Ridge(alpha=1.0)),
])

pipe_ridge.fit(X_train, y_train)

print('=== Ridge Regression ===')
print(f'Train RMSE : {np.sqrt(mean_squared_error(y_train, pipe_ridge.predict(X_train))):.4f}')
print(f'Val   RMSE : {np.sqrt(mean_squared_error(y_val,   pipe_ridge.predict(X_val))):.4f}')

---
# POLYNOMIAL REGRESSION
Use when plots in step 3 showed a curved relationship

## 5b. Pipeline — Scale + Poly + Ridge

In [ ]:
# always pair poly with Ridge — poly adds many features → overfitting risk
pipe_poly = Pipeline([
    ('scale', StandardScaler()),
    ('poly',  PolynomialFeatures(degree=2, include_bias=False)),
    ('model', Ridge(alpha=1.0)),
])

pipe_poly.fit(X_train, y_train)

y_pred_poly_train = pipe_poly.predict(X_train)
y_pred_poly_val   = pipe_poly.predict(X_val)

print('=== Polynomial + Ridge ===')
print(f'Train RMSE : {np.sqrt(mean_squared_error(y_train, y_pred_poly_train)):.4f}')
print(f'Val   RMSE : {np.sqrt(mean_squared_error(y_val,   y_pred_poly_val)):.4f}')

## 6b. Plot Predicted vs Actual

In [ ]:
plt.scatter(y_val, y_pred_poly_val, alpha=0.5)
plt.plot([y_val.min(), y_val.max()],
         [y_val.min(), y_val.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Polynomial + Ridge — Actual vs Predicted')
plt.show()

## 7b. Check Overfitting / Underfitting

In [ ]:
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_poly_train))
val_rmse   = np.sqrt(mean_squared_error(y_val,   y_pred_poly_val))

print(f'Train RMSE : {train_rmse:.4f}')
print(f'Val   RMSE : {val_rmse:.4f}')
print()

if train_rmse > 0.3 and val_rmse > 0.3:
    print('→ HIGH BIAS (underfitting) — try degree=3')
elif val_rmse > train_rmse * 1.5:
    print('→ HIGH VARIANCE (overfitting) — increase alpha')
else:
    print('→ Good fit')

## 8b. Tune alpha and degree

In [ ]:
# manually try combinations and pick the one with lowest val RMSE
for degree in [2, 3]:
    for alpha in [0.01, 0.1, 1.0, 10.0]:
        pipe = Pipeline([
            ('scale', StandardScaler()),
            ('poly',  PolynomialFeatures(degree=degree, include_bias=False)),
            ('model', Ridge(alpha=alpha)),
        ])
        pipe.fit(X_train, y_train)
        val_rmse = np.sqrt(mean_squared_error(y_val, pipe.predict(X_val)))
        print(f'degree={degree}  alpha={alpha:<6}  val RMSE={val_rmse:.4f}')

---
# LOGISTIC REGRESSION
Use when target is binary (0 / 1) — e.g. Titanic survived

## 4c. Prepare X and y for Classification

In [ ]:

target_clf = 'target_binary'   # target coloumn

X_c = df[feature_cols].values
y_c = df[target_clf].values

X_train_c, X_tmp_c,  y_train_c, y_tmp_c  = train_test_split(X_c, y_c, test_size=0.30, random_state=42)
X_val_c,   X_test_c, y_val_c,   y_test_c = train_test_split(X_tmp_c, y_tmp_c, test_size=0.50, random_state=42)

## 5c. Pipeline — Scale + Logistic Regression

In [ ]:
# C = 1/lambda → smaller C = more regularization (opposite of Ridge alpha)
# try: 0.01, 0.1, 1.0, 10.0

pipe_clf = Pipeline([
    ('scale', StandardScaler()),
    ('model', LogisticRegression(C=1.0, max_iter=1000)),
])

pipe_clf.fit(X_train_c, y_train_c)

y_pred_clf  = pipe_clf.predict(X_val_c)              # 0 or 1
y_proba_clf = pipe_clf.predict_proba(X_val_c)[:, 1]  # probability of class 1

print('=== Logistic Regression ===')
print(f'Val Accuracy : {accuracy_score(y_val_c, y_pred_clf):.4f}')

## 6c. Plot Predicted Probabilities

In [ ]:
plt.hist(y_proba_clf[y_val_c == 0], bins=20, alpha=0.6, label='Actual 0')
plt.hist(y_proba_clf[y_val_c == 1], bins=20, alpha=0.6, label='Actual 1')
plt.axvline(0.5, color='red', linestyle='--', label='threshold 0.5')
plt.xlabel('Predicted Probability')
plt.ylabel('Count')
plt.title('Logistic Regression — Predicted Probabilities')
plt.legend()
plt.show()

# two separate humps on each side = model is confident and separating well
# overlapping humps = model is confused

## 7c. Check Overfitting / Underfitting

In [ ]:
train_acc = accuracy_score(y_train_c, pipe_clf.predict(X_train_c))
val_acc   = accuracy_score(y_val_c,   y_pred_clf)

print(f'Train Accuracy : {train_acc:.4f}')
print(f'Val   Accuracy : {val_acc:.4f}')
print()

if train_acc < 0.75 and val_acc < 0.75:
    print('→ HIGH BIAS (underfitting) — add more features or increase C')
elif train_acc - val_acc > 0.1:
    print('→ HIGH VARIANCE (overfitting) — decrease C')
else:
    print('→ Good fit')

## 8c. Tune C (regularization)

In [ ]:
for C in [0.01, 0.1, 1.0, 10.0, 100.0]:
    pipe = Pipeline([
        ('scale', StandardScaler()),
        ('model', LogisticRegression(C=C, max_iter=1000)),
    ])
    pipe.fit(X_train_c, y_train_c)
    val_acc = accuracy_score(y_val_c, pipe.predict(X_val_c))
    print(f'C={C:<8}  val accuracy={val_acc:.4f}')

---
## 9. Final Evaluation on Test Set
Only run this once — after you have picked your best model above

In [ ]:
# regression
test_rmse = np.sqrt(mean_squared_error(y_test, pipe_ridge.predict(X_test)))
print(f'Test RMSE : {test_rmse:.4f}')

# classification
test_acc = accuracy_score(y_test_c, pipe_clf.predict(X_test_c))
print(f'Test Accuracy : {test_acc:.4f}')

# test result should be close to val result
# if test is much worse → you overfit on val by tuning too much